In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 입력 파일
IN_FILES = [
    "../data/clean_hyperblick.csv",
    "../data/clean_koreamasa.csv",
]

SIGUNGU_FILE = "../data/sigungu_code_info.csv"
STATION_FILE = "../data/서울교통공사_역주소 및 전화번호_20250318.csv"

TARGET_COLS = [
    "name",
    "category",
    "address_std",
    "gu",
    "sigu",
    "lat",
    "lng",
    "is_valid_location",
    "COL_ADM_SE",
    "station",
    "station_exit",
    "station_detail",
]

In [2]:
def read_csv_smart(path):
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, encoding="latin1")


def normalize_str(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None


def is_valid_seoul_latlng(lat, lng):
    try:
        if pd.isna(lat) or pd.isna(lng):
            return False
        lat = float(lat)
        lng = float(lng)
    except Exception:
        return False

    return (37.3 <= lat <= 37.8) and (126.7 <= lng <= 127.3)

In [3]:
# 시군구 코드 매핑
sigungu = read_csv_smart(SIGUNGU_FILE)
sigungu.columns = sigungu.columns.str.strip()

gu_to_code = dict(zip(
    sigungu["SIGUNGU_NM"].astype(str),
    sigungu["SIGUNGU_CD"].astype(str)
))

# 역 정보
st = read_csv_smart(STATION_FILE)
st.columns = st.columns.str.strip()

station_names = (
    st["역명"]
    .astype(str)
    .str.strip()
    .replace("", np.nan)
    .dropna()
    .unique()
    .tolist()
)

station_tokens = [f"{nm}역" for nm in station_names]

In [7]:
def parse_station(name, addr_std):
    name_s = normalize_str(name) or ""
    addr_s = normalize_str(addr_std) or ""

    # address 우선 탐색
    for tok in station_tokens:
        if tok in addr_s:
            return tok.replace("역", ""), "address_contains_station"

    for tok in station_tokens:
        if tok in name_s:
            return tok.replace("역", ""), "name_contains_station"

    return None, None


In [8]:
def align_to_fclean(path):
    df = read_csv_smart(path)
    df.columns = df.columns.str.strip()

    # address_std 보정
    if "address_std" not in df.columns:
        df["address_std"] = df["address"] if "address" in df.columns else None

    for c in ["name", "gu", "lat", "lng"]:
        if c not in df.columns:
            df[c] = None

    if "category" not in df.columns:
        df["category"] = None

    df["sigu"] = df["gu"].apply(
    lambda x: f"서울특별시 {str(x).strip()}" if pd.notna(x) and str(x).strip() != "" else None
)

    # COL_ADM_SE 채우기
    df["COL_ADM_SE"] = df["gu"].map(
        lambda x: gu_to_code.get(str(x).strip()) if pd.notna(x) else None
    )

    # 역 정보 채우기
    stations = df.apply(
        lambda r: parse_station(r.get("name"), r.get("address_std")),
        axis=1,
        result_type="expand"
    )

    df["station"] = stations[0]
    df["station_detail"] = stations[1]
    df["station_exit"] = None

    # 위치 유효성
    df["is_valid_location"] = df.apply(
        lambda r: is_valid_seoul_latlng(r.get("lat"), r.get("lng")),
        axis=1
    )

    df["address_std"] = df["address_std"].map(normalize_str)

    # 스키마 맞추기
    for col in TARGET_COLS:
        if col not in df.columns:
            df[col] = None

    df = df[TARGET_COLS]

    return df

In [10]:
for in_path in IN_FILES:
    out_df = align_to_fclean(in_path)

    p = Path(in_path)
    out_path = p.with_name("f_clean_" + p.name)

    out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"[OK] 저장 완료 → {out_path}")
    print("rows:", len(out_df))
    print("COL_ADM_SE null ratio:", out_df["COL_ADM_SE"].isna().mean())
    print("station filled ratio:", out_df["station"].notna().mean())
    print("valid location ratio:", out_df["is_valid_location"].mean())
    print("-" * 50)

[OK] 저장 완료 → ..\data\f_clean_clean_hyperblick.csv
rows: 7
COL_ADM_SE null ratio: 0.0
station filled ratio: 0.0
valid location ratio: 1.0
--------------------------------------------------
[OK] 저장 완료 → ..\data\f_clean_clean_koreamasa.csv
rows: 3
COL_ADM_SE null ratio: 0.0
station filled ratio: 0.0
valid location ratio: 1.0
--------------------------------------------------
